# 3 — Baselines Unificados: DummyClassifier + Regressão Logística
**Etapa 1 · Tech Challenge POSTECH MLOps**

---

## Sumário Técnico

Este notebook consolida os experimentos de baseline em um único artefato reprodutível,
cobrindo dois modelos de referência para o problema de **previsão de churn em telecomunicações**.

### Objetivo
Estabelecer um **piso** (Dummy) e um **teto linear** (Logística) de performance antes
da etapa de modelagem com MLP/PyTorch. Ambos os modelos são registrados no MLflow com
dataset versionado (URI + MD5), parâmetros, métricas técnicas e de negócio.

### Fluxo de execução

```
data/processed/
├── train.parquet   
└── test.parquet    
         │
         ▼
┌─────────────────────────────────────────────┐
│  Modelo A: DummyClassifier (most_frequent)  │  piso absoluto
│  Modelo B: LogisticRegression (lbfgs)       │  teto linear
└─────────────────────────────────────────────┘
         │
         ├── Validação cruzada estratificada 5-fold (X_train)
         ├── Avaliação hold-out única (X_test — nunca usado no dev)
         ├── Métricas técnicas: AUC-ROC, PR-AUC, F1, Recall, Precision
         ├── Métricas de negócio: Custo Total, Receita Salva, Valor Líquido
         ├── Threshold ótimo por valor de negócio (seção 4.7)
         └── Registro MLflow (SQLite) + figuras em reports/figures/baselines/
```

### Decisões técnicas

| Decisão | Justificativa |
|---|---|
| `class_weight="balanced"` | Test set tem desbalanceamento residual (~26% churn) |
| `solver="lbfgs"` | Eficiente para datasets de tamanho médio (~5-6k amostras) |
| `max_iter=1000` | Garante convergência sem warning |
| CV estratificada k=5 | Preserva proporção de churn em cada fold |
| `matplotlib.use('Agg')` | Renderização headless (sem display no servidor) |

### Métricas de negócio (trade-off FP × FN)

| Conceito | Impacto | Custo (config.py) |
|---|---|---|
| **Falso Negativo (FN)** | Perder cliente sem tentar reter | `COST_FN = R$ 2.845,00` |
| **Falso Positivo (FP)** | Campanha de retenção desperdiçada | `COST_FP = R$ 73,52` |
| **Verdadeiro Positivo (TP)** | Cliente retido (salva CLV - custo retenção) | `COST_CLV - COST_RETENTION` |

### Rastreamento MLflow

Backend **SQLite** (`mlflow.db`) configurado em `churn_telecom/config.py`:
```python
MLFLOW_TRACKING_URI = f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}"
MLFLOW_ARTIFACT_URI = (PROJECT_ROOT / 'mlartifacts').as_uri()
```
Ambos ignorados pelo `.gitignore` e as figuras em `reports/figures/baselines/` **são commitadas** no git.

### Localização dos artefatos

| Artefato | Path | Git |
|---|---|---|
| Confusion matrix (Dummy) | `reports/figures/baselines/dummy_confusion_matrix.png` | ✅ |
| ROC curve (Dummy) | `reports/figures/baselines/dummy_roc_curve.png` | ✅ |
| Confusion matrix (Logística) | `reports/figures/baselines/logistic_confusion_matrix.png` | ✅ |
| ROC curve (Logística) | `reports/figures/baselines/logistic_roc_curve.png` | ✅ |
| PR-AUC curve (Logística) | `reports/figures/baselines/logistic_pr_curve.png` | ✅ |
| Feature importance (Logística) | `reports/figures/baselines/logistic_feature_importance.png` | ✅ |


---
## 0. Imports e Setup

Importa todas as dependências do projeto. Pontos de atenção:
- `matplotlib.use('Agg')` **antes** de qualquer `import matplotlib.pyplot` — garante renderização headless
- `np.random.seed(RANDOM_STATE)` — seed global para reprodutibilidade de operações NumPy
- `get_logger()` de `config.py` — logging estruturado com handler de arquivo + console (sem `print()`)
- `warnings.filterwarnings('ignore')` — suprime UserWarnings do sklearn em notebooks


In [128]:
"""Imports, seed global e configuração de logging via config.py."""
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless — deve vir antes de qualquer plt import

import numpy as np
import pandas as pd

# ── MLflow ────────────────────────────────────────────────────────────────────
import mlflow
import mlflow.sklearn

# ── Scikit-Learn ──────────────────────────────────────────────────────────────
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

# ── Resolve raiz do projeto (notebooks/ está um nível abaixo) ─────────────────
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Configuração central — única fonte de verdade ─────────────────────────────
from churn_telecom.config import (
    COST_CLV,
    COST_FN,
    COST_FP,
    COST_RETENTION,
    DATA_PROCESSED,
    MLFLOW_EXPERIMENT,
    MODELS_DIR,
    PROJECT_ROOT,
    RANDOM_STATE,
    REPORTS_FIGURES,
    TARGET,
    TARGET_COL,
    get_logger,
    log_dataset_to_mlflow,
    mlflow_run,
)

# ── Visualização ──────────────────────────────────────────────────────────────
from churn_telecom.plots import (
    save_confusion_matrix,
    save_feature_importance,
    save_precision_recall_curve,
    save_roc_curve,
    save_threshold_f1_recall,
)

# ── Seed global ───────────────────────────────────────────────────────────────
np.random.seed(RANDOM_STATE)

# ── Logger estruturado — sem print() ─────────────────────────────────────────
logger = get_logger("3_baselines_unificado")
logger.info("ROOT            : %s", ROOT)
logger.info("PROJECT_ROOT    : %s", PROJECT_ROOT)
logger.info("RANDOM_STATE    : %d", RANDOM_STATE)
logger.info("MLflow version  : %s", mlflow.__version__)
logger.info("COST_FN=%.2f | COST_FP=%.2f | COST_CLV=%.2f | COST_RETENTION=%.2f",
            COST_FN, COST_FP, COST_CLV, COST_RETENTION)

warnings.filterwarnings("ignore", category=UserWarning)

15:28:12 | INFO | ROOT            : c:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\tech_challenge\tech_challenge_1_fiap_mlops
15:28:12 | INFO | PROJECT_ROOT    : C:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\tech_challenge\tech_challenge_1_fiap_mlops
15:28:12 | INFO | RANDOM_STATE    : 42
15:28:12 | INFO | MLflow version  : 3.11.1
15:28:12 | INFO | COST_FN=2845.00 | COST_FP=73.52 | COST_CLV=2845.00 | COST_RETENTION=73.52


---
## 1. Paths e Pré-requisitos

Todos os paths são derivados de `config.py` — sem strings hardcoded no notebook.  
A célula valida a existência dos artefatos obrigatórios (train e test) **antes** de executar qualquer treinamento, interrompendo com erro descritivo se algum estiver faltando.


In [129]:
# ── Inputs ────────────────────────────────────────────────────────────────────
TRAIN_PATH = DATA_PROCESSED / "train.parquet"
TEST_PATH  = DATA_PROCESSED / "test.parquet"

# ── Outputs: figuras commitadas no git ────────────────────────────────────────
FIGURES_DIR = REPORTS_FIGURES / "baselines"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Validação de pré-requisitos ───────────────────────────────────────────────
_required = [TRAIN_PATH, TEST_PATH]
_missing  = [p for p in _required if not p.exists()]

if _missing:
    for p in _missing:
        logger.error("FALTANDO: %s", p.relative_to(PROJECT_ROOT))
    raise FileNotFoundError(
        "Execute os notebooks 1 → 2 antes deste. "
        f"Faltando: {[str(p.relative_to(PROJECT_ROOT)) for p in _missing]}"
    )

for p in _required:
    logger.info("OK  %s", p.relative_to(PROJECT_ROOT))

logger.info("FIGURES_DIR : %s", FIGURES_DIR.relative_to(PROJECT_ROOT))

15:28:12 | INFO | OK  data\processed\train.parquet
15:28:12 | INFO | OK  data\processed\test.parquet
15:28:12 | INFO | FIGURES_DIR : reports\figures\baselines


---
## 2. Carregamento dos Dados

| Split | Origem | Tamanho esperado | Churn esperado |
|---|---|---|---|
| `train.parquet` | `data/processed/` | ~5.6k amostras | ~26% |
| `test.parquet` | `data/processed/` | ~1.4k amostras | ~26% |

> ⚠️ O **test set nunca é usado no desenvolvimento** — apenas na avaliação final de cada modelo (uso único por run).


In [130]:
# ── Carrega dados já transformados pelo notebook 1.03 ─────────────────────────
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

X_train = train.drop(columns=[TARGET_COL])
y_train = train[TARGET_COL].astype(int)
X_test  = test.drop(columns=[TARGET_COL])
y_test  = test[TARGET_COL].astype(int)

feature_names: list[str] = X_train.columns.tolist()

logger.info("Train: %d × %d | Churn=%.2f%%", *X_train.shape, y_train.mean() * 100)
logger.info("Test : %d × %d | Churn=%.2f%%", *X_test.shape,  y_test.mean()  * 100)
logger.info("Features (%d): %s", len(feature_names), feature_names)

15:28:12 | INFO | Train: 5616 × 30 | Churn=26.44%
15:28:12 | INFO | Test : 1405 × 30 | Churn=26.48%
15:28:12 | INFO | Features (30): ['tenure_months', 'monthly_charges', 'total_charges', 'num_services', 'charges_per_month', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'paperless_billing', 'senior_citizen', 'internet_service_Fiber optic', 'internet_service_No', 'contract_One year', 'contract_Two year', 'payment_method_Credit card (automatic)', 'payment_method_Electronic check', 'payment_method_Mailed check', 'gender_Male', 'tenure_group_medio', 'tenure_group_novo', 'is_month_to_month', 'has_security_support', 'is_fiber_optic']


---
## 3. Utilitários compartilhados

Funções auxiliares reutilizadas por ambos os modelos:

- **`compute_holdout_metrics`** — calcula o dicionário completo de métricas do hold-out (AUC-ROC, PR-AUC, F1, Recall, Precision, Accuracy, TP/TN/FP/FN)
- **`compute_business_metrics`** — traduz TP/FP/FN para R$ usando as constantes de custo de `config.py`
- **`find_best_threshold_business`** — busca exaustiva do threshold que **maximiza o valor líquido de negócio**

> 📌 `find_best_threshold_business` implementa a lógica de negócio central do projeto:  
> cada threshold gera uma combinação de TP/FP/FN → cada combinação tem um valor monetário → escolhemos o threshold que **maximiza o valor líquido**.


In [131]:
"""Utilitários: métricas técnicas, métricas de negócio e otimização de threshold."""


def compute_holdout_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> dict[str, float]:
    """Retorna dicionário completo de métricas do hold-out.

    Nomes espelham os gerados pelo sklearn em cross_validate (prefixo 'test_')
    para facilitar comparação direta com os resultados de CV.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "test_roc_auc"           : round(float(roc_auc_score(y_true, y_proba)), 2),
        "test_average_precision" : round(float(average_precision_score(y_true, y_proba)), 2),
        "test_f1"                : round(float(f1_score(y_true, y_pred, zero_division=0)), 2),
        "test_recall"            : round(float(recall_score(y_true, y_pred, zero_division=0)), 2),
        "test_precision"         : round(float(precision_score(y_true, y_pred, zero_division=0)), 2),
        "test_accuracy"          : round(float((tp + tn) / len(y_true)), 2),
        # Células da matriz — necessárias para cálculo de custo de negócio
        "test_tp" : round(float(tp), 2),
        "test_tn" : round(float(tn), 2),
        "test_fp" : round(float(fp), 2),
        "test_fn" : round(float(fn), 2),
    }


def compute_business_metrics(
    tp: float,
    fp: float,
    fn: float,
    cost_fn: float = COST_FN,
    cost_fp: float = COST_FP,
    cost_clv: float = COST_CLV,
    cost_retention: float = COST_RETENTION,
) -> dict[str, float]:
    """Traduz TP/FP/FN para métricas monetárias.

    Lógica:
    - TP: acertamos o churner → podemos tentar reter → economizamos (CLV - custo_retenção)
    - FP: alarmamos incorretamente → gastamos custo_retenção desnecessariamente
    - FN: perdemos o churner sem chance de retenção → perdemos o CLV completo
    """
    receita_salva   = (cost_clv - cost_retention) * tp
    custo_fp        = cost_fp * fp
    custo_fn        = cost_fn * fn
    custo_total     = custo_fp + custo_fn
    valor_liquido   = receita_salva - custo_total
    return {
        "biz_receita_salva"   : round(receita_salva, 2),
        "biz_custo_fp"        : round(custo_fp, 2),
        "biz_custo_fn"        : round(custo_fn, 2),
        "biz_custo_total"     : round(custo_total, 2),
        "biz_valor_liquido"   : round(valor_liquido, 2),
    }


def find_best_threshold_business(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    cost_fn: float = COST_FN,
    cost_fp: float = COST_FP,
    cost_clv: float = COST_CLV,
    cost_retention: float = COST_RETENTION,
    n_thresholds: int = 200,
) -> tuple[float, float, list[tuple]]:
    """Busca exaustiva do threshold que maximiza o valor líquido de negócio.

    NOTA: Esta é uma implementação local. Quando a função externa de threshold
    (threshold_utils.py) for integrada ao projeto, substitua esta por:
        from churn_telecom.threshold_utils import find_best_threshold_business

    Parâmetros
    ----------
    y_true         : labels verdadeiros (0/1)
    y_proba        : probabilidades preditas da classe positiva
    n_thresholds   : número de pontos na grade de busca [0.01, 0.99]

    Retorna
    -------
    best_threshold : threshold ótimo
    best_value     : valor líquido máximo (R$)
    results        : lista de (threshold, valor_liquido, tp, fp, fn, tn)
    """
    thresholds = np.linspace(0.01, 0.99, n_thresholds)
    best_threshold = 0.5
    best_value = -np.inf
    results: list[tuple] = []

    for thr in thresholds:
        y_pred_thr = (y_proba >= thr).astype(int)
        tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, y_pred_thr).ravel()
        biz = compute_business_metrics(tp_t, fp_t, fn_t, cost_fn, cost_fp, cost_clv, cost_retention)
        net = biz["biz_valor_liquido"]
        results.append((thr, net, int(tp_t), int(fp_t), int(fn_t), int(tn_t)))
        if net > best_value:
            best_value = net
            best_threshold = float(thr)

    logger.info(
        "find_best_threshold_business | best_thr=%.4f | best_value=R$%.2f",
        best_threshold, best_value,
    )
    return best_threshold, best_value, results


# ── Scorers para cross_validate ───────────────────────────────────────────────
# Nomes canônicos do sklearn → chaves geradas: cv_<scorer>_mean / _std
SCORING: list[str] = [
    "roc_auc",
    "average_precision",   # PR-AUC (nome canônico do sklearn)
    "f1",
    "recall",
    "precision"
]

# ── StratifiedKFold compartilhado ─────────────────────────────────────────────
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

logger.info("Utilitários carregados | SCORING=%s | CV=%s", SCORING, CV)

15:28:12 | INFO | Utilitários carregados | SCORING=['roc_auc', 'average_precision', 'f1', 'recall', 'precision'] | CV=StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


---
## 4. Modelo A — DummyClassifier (`most_frequent`)

O `DummyClassifier` estabelece o **piso absoluto** de performance:
- Prediz **sempre** a classe majoritária (`churn_value = 0`, ~73.5% do test set)
- Não lê nenhuma feature — ignora completamente os dados de entrada
- Revela o perigo de usar **acurácia** como métrica: atinge ~73.5% sem aprender nada
- **100% de Falsos Negativos** → perda máxima de negócio (nenhum churner é identificado)

Todo modelo subsequente deve superar este baseline em **AUC-ROC, PR-AUC e Valor Líquido de Negócio**.


### 4.1 Definição do Pipeline

Os parâmetros são declarados em um dicionário explícito (`DUMMY_PARAMS`) para garantir
rastreabilidade no MLflow — `mlflow.log_params()` receberá exatamente este dicionário.

> `SelectKBest` **não é aplicado** aqui — o Dummy ignora features por definição.


In [132]:
# Parâmetros do DummyClassifier — declarados explicitamente para logging no MLflow
DUMMY_PARAMS: dict = {
    "strategy"     : "most_frequent",
    "random_state" : RANDOM_STATE,
}

logger.info("DUMMY_PARAMS: %s", DUMMY_PARAMS)

# Pipeline simples: dados já estão transformados (preprocessor aplicado em 1.03)
dummy_pipeline = Pipeline(
    [("classifier", DummyClassifier(**DUMMY_PARAMS))]
)

15:28:12 | INFO | DUMMY_PARAMS: {'strategy': 'most_frequent', 'random_state': 42}


### 4.2 Validação Cruzada Estratificada (5-fold)

Roda `cross_validate` sobre `X_train` com `StratifiedKFold(k=5)` compartilhado.

> **Mapeamento sklearn → chave no MLflow**
>
> | Conceito | `scoring=` (sklearn) | Chave logada no MLflow |
> |---|---|---|
> | AUC-ROC | `"roc_auc"` | `cv_roc_auc_mean` / `_std` |
> | PR-AUC | `"average_precision"` | `cv_average_precision_mean` / `_std` |
> | F1 | `"f1"` | `cv_f1_mean` / `_std` |
> | Recall | `"recall"` | `cv_recall_mean` / `_std` |
> | Precision | `"precision"` | `cv_precision_mean` / `_std` |
> | MCC | `"matthews_corrcoef"` | `cv_matthews_corrcoef_mean` / `_std` |
>
> ⚠️ `cv_pr_auc_mean` **não existe** — o sklearn nomeia PR-AUC como `average_precision`.


In [133]:
logger.info(
    "cross_validate [Dummy] | strategy=%s | folds=%d | stratified=True",
    DUMMY_PARAMS["strategy"],
    CV.n_splits,
)

dummy_cv_results = cross_validate(
    dummy_pipeline,
    X_train,
    y_train,
    cv=CV,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)

# Sumário CV — dicionário {scorer: {mean, std}}
dummy_cv_summary: dict[str, dict[str, float]] = {}
for key, values in dummy_cv_results.items():
    if key.startswith("test_"):
        name = key.removeprefix("test_")
        dummy_cv_summary[name] = {
            "mean" : float(values.mean()),
            "std"  : float(values.std()),
        }
        logger.info(
            "  cv [Dummy] %-26s  mean=%+.4f  std=%.4f",
            name, values.mean(), values.std(),
        )

15:28:12 | INFO | cross_validate [Dummy] | strategy=most_frequent | folds=5 | stratified=True
15:28:29 | INFO |   cv [Dummy] roc_auc                     mean=+0.5000  std=0.0000
15:28:29 | INFO |   cv [Dummy] average_precision           mean=+0.2644  std=0.0001
15:28:29 | INFO |   cv [Dummy] f1                          mean=+0.0000  std=0.0000
15:28:29 | INFO |   cv [Dummy] recall                      mean=+0.0000  std=0.0000
15:28:29 | INFO |   cv [Dummy] precision                   mean=+0.0000  std=0.0000


### 4.3 Avaliação no Hold-out

Treina o pipeline no `X_train` completo e avalia **uma única vez** no `X_test`.
O test set **nunca foi visto** durante o desenvolvimento — este é o resultado final do Dummy.

Métricas esperadas para `most_frequent` com ~26% de churn:
- `AUC-ROC ≈ 0.500` — equivalente ao acaso
- `PR-AUC ≈ 0.265` — igual à taxa base de churn (piso da PR-AUC)
- `Recall = 0.000` — nenhum churner detectado → 100% de FN
- `NPV ≈ 0.735` — proporção de verdadeiros negativos


In [134]:
# Treino final no X_train completo
dummy_pipeline.fit(X_train, y_train)

dummy_y_pred  = dummy_pipeline.predict(X_test)
dummy_y_proba = dummy_pipeline.predict_proba(X_test)[:, 1]

dummy_hold_out = compute_holdout_metrics(y_test.values, dummy_y_pred, dummy_y_proba)

for k, v in dummy_hold_out.items():
    logger.info("  hold-out [Dummy] %-30s  %.4f", k, v)

15:28:29 | INFO |   hold-out [Dummy] test_roc_auc                    0.5000
15:28:29 | INFO |   hold-out [Dummy] test_average_precision          0.2600
15:28:29 | INFO |   hold-out [Dummy] test_f1                         0.0000
15:28:29 | INFO |   hold-out [Dummy] test_recall                     0.0000
15:28:29 | INFO |   hold-out [Dummy] test_precision                  0.0000
15:28:29 | INFO |   hold-out [Dummy] test_accuracy                   0.7400
15:28:29 | INFO |   hold-out [Dummy] test_tp                         0.0000
15:28:29 | INFO |   hold-out [Dummy] test_tn                         1033.0000
15:28:29 | INFO |   hold-out [Dummy] test_fp                         0.0000
15:28:29 | INFO |   hold-out [Dummy] test_fn                         372.0000


### 4.4 Métricas de Negócio — Dummy

Traduz TP/FP/FN para impacto monetário usando as constantes de `config.py`:

| Variável | Valor | Significado |
|---|---|---|
| `COST_FN` | R$ 2.845,00 | Receita perdida por churner não identificado |
| `COST_FP` | R$ 73,52 | Custo de campanha de retenção desnecessária |
| `COST_CLV` | R$ 2.845,00 | Receita anual média por cliente |
| `COST_RETENTION` | R$ 73,52 | Custo de uma ação de retenção |

Como o Dummy prediz **sempre negativo** (nenhum churner identificado), esperamos:
- `TP = 0` → `receita_salva = R$ 0`
- `FN = todos os churners` → custo máximo de FN
- Valor líquido **negativo e máximo** — pior caso possível


In [135]:
dummy_biz = compute_business_metrics(
    tp=dummy_hold_out["test_tp"],
    fp=dummy_hold_out["test_fp"],
    fn=dummy_hold_out["test_fn"],
)

logger.info("Métricas de negócio [Dummy]:")
for k, v in dummy_biz.items():
    logger.info("  %-35s : R$ %.2f", k, v)

15:28:29 | INFO | Métricas de negócio [Dummy]:
15:28:29 | INFO |   biz_receita_salva                   : R$ 0.00
15:28:29 | INFO |   biz_custo_fp                        : R$ 0.00
15:28:29 | INFO |   biz_custo_fn                        : R$ 1058340.00
15:28:29 | INFO |   biz_custo_total                     : R$ 1058340.00
15:28:29 | INFO |   biz_valor_liquido                   : R$ -1058340.00


### 4.5 Figuras — Dummy

Figuras salvas em `reports/figures/baselines/` (**commitadas no git**) antes de serem
logadas no MLflow. Isso garante disponibilidade no repositório sem executar o MLflow.

```
reports/figures/baselines/     ← git
    dummy_confusion_matrix.png
    dummy_roc_curve.png

mlartifacts/<exp_id>/<run_id>/figures/   ← local only (cópia)
```


In [136]:
# ── 4.5.1 Confusion Matrix ────────────────────────────────────────────────────
dummy_cm_path = save_confusion_matrix(
    y_true=y_test,
    y_pred=dummy_y_pred,
    path=FIGURES_DIR / "dummy_confusion_matrix.png",
    title="Confusion Matrix — DummyClassifier (most_frequent)",
)

# ── 4.5.2 ROC Curve ───────────────────────────────────────────────────────────
dummy_roc_path = save_roc_curve(
    y_true=y_test,
    y_proba=dummy_y_proba,
    path=FIGURES_DIR / "dummy_roc_curve.png",
    model_name="DummyClassifier",
    title=f"ROC Curve — DummyClassifier  (AUC = {dummy_hold_out['test_roc_auc']:.3f})",
)

logger.info("Figuras [Dummy] salvas em: %s", FIGURES_DIR.relative_to(PROJECT_ROOT))

15:28:30 | INFO | Figuras [Dummy] salvas em: reports\figures\baselines


### 4.6 Registro no MLflow — Dummy

Usa o context manager `mlflow_run()` de `config.py`. O que é registrado:

1. **Tags** — notebook, fase, modelo, framework
2. **Dataset** — `log_dataset_to_mlflow()` com URI + hash MD5 do parquet (train + test)
3. **Parâmetros** — modelo, CV, dados, constantes de custo
4. **Métricas CV** — `cv_<scorer>_mean` / `_std` (6 scorers × 2 = 12 métricas)
5. **Métricas hold-out** — `test_<scorer>` (12 métricas técnicas)
6. **Métricas de negócio** — `biz_*` (5 métricas monetárias)
7. **Artefatos** — cópias das figuras salvas em `reports/figures/baselines/`
8. **Modelo** — pipeline serializado via `mlflow.sklearn.log_model` (sem registry no baseline)


In [137]:
with mlflow_run(run_name="dummy_most_frequent") as run_dummy:
    DUMMY_RUN_ID = run_dummy.info.run_id
    DUMMY_EXP_ID = run_dummy.info.experiment_id
    logger.info("Run [Dummy] iniciado | run_id=%s | exp_id=%s", DUMMY_RUN_ID, DUMMY_EXP_ID)

    # ── 1. Tags ───────────────────────────────────────────────────────────────
    mlflow.set_tags({
        "notebook" : "3.00",
        "fase"     : "baseline",
        "modelo"   : "dummy",
        "task"     : "classification",
        "framework": "sklearn",
    })

    # ── 2. Dataset versionado (URI + MD5) ─────────────────────────────────────
    log_dataset_to_mlflow(X_train, y_train, split="train", source_path=TRAIN_PATH)
    log_dataset_to_mlflow(X_test,  y_test,  split="test",  source_path=TEST_PATH)
    logger.info("Datasets logados no MLflow com URI")

    # ── 3. Parâmetros ─────────────────────────────────────────────────────────
    mlflow.log_params({
        # Modelo
        "model_type"        : "DummyClassifier",
        **DUMMY_PARAMS,
        "feature_selection" : "none",
        # Validação
        "cv_folds"          : CV.n_splits,
        "cv_stratified"     : True,
        "cv_scoring"        : str(SCORING),
        # Dados
        "dataset_train"     : str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
        "dataset_test"      : str(TEST_PATH.relative_to(PROJECT_ROOT)),
        "n_train"           : int(len(X_train)),
        "n_test"            : int(len(X_test)),
        "n_features"        : int(X_train.shape[1]),
        "churn_rate_train"  : round(float(y_train.mean()), 4),
        "churn_rate_test"   : round(float(y_test.mean()),  4),
        "target_col"        : TARGET,
        # Custos de negócio
        "cost_fn"        : COST_FN,
        "cost_fp"        : COST_FP,
        "cost_clv"       : COST_CLV,
        "cost_retention" : COST_RETENTION,
    })
    logger.info("Parâmetros logados")

    # ── 4. Métricas CV ────────────────────────────────────────────────────────
    for scorer_name, stats in dummy_cv_summary.items():
        mlflow.log_metric(f"cv_{scorer_name}_mean", stats["mean"])
        mlflow.log_metric(f"cv_{scorer_name}_std",  stats["std"])
    logger.info("Métricas CV logadas (%d scorers × 2)", len(dummy_cv_summary))

    # ── 5. Métricas hold-out técnicas ─────────────────────────────────────────
    for metric_name, metric_value in dummy_hold_out.items():
        mlflow.log_metric(metric_name, metric_value)
    logger.info("Métricas hold-out logadas (%d)", len(dummy_hold_out))

    # ── 6. Métricas de negócio ────────────────────────────────────────────────
    for metric_name, metric_value in dummy_biz.items():
        mlflow.log_metric(metric_name, metric_value)
    logger.info("Métricas de negócio logadas (%d)", len(dummy_biz))

    # ── 7. Artefatos ──────────────────────────────────────────────────────────
    for fig_path in [dummy_cm_path, dummy_roc_path]:
        mlflow.log_artifact(str(fig_path), artifact_path="figures")
        logger.info("Artefato logado: %s → mlartifacts/.../figures/", fig_path.name)

    # ── 8. Modelo ─────────────────────────────────────────────────────────────
    mlflow.sklearn.log_model(
        sk_model=dummy_pipeline,
        artifact_path="dummy_classifier",
        registered_model_name=None,  # sem registry — apenas baseline de referência
    )

    logger.info(
        "Run [Dummy] finalizado | experiment=%s | run_id=%s",
        MLFLOW_EXPERIMENT, DUMMY_RUN_ID,
    )
    logger.info(
        "Resumo [Dummy] | roc_auc=%.4f | avg_precision=%.4f | f1=%.4f | recall=%.4f | valor_liquido=R$%.2f",
        dummy_hold_out["test_roc_auc"],
        dummy_hold_out["test_average_precision"],
        dummy_hold_out["test_f1"],
        dummy_hold_out["test_recall"],
        dummy_biz["biz_valor_liquido"],
    )

15:28:30 | INFO | Run [Dummy] iniciado | run_id=64f33f3553594287a325fefe5040bedf | exp_id=1
15:28:30 | INFO | Datasets logados no MLflow com URI
15:28:30 | INFO | Parâmetros logados
15:28:30 | INFO | Métricas CV logadas (5 scorers × 2)
15:28:30 | INFO | Métricas hold-out logadas (10)
15:28:30 | INFO | Métricas de negócio logadas (5)
15:28:30 | INFO | Artefato logado: dummy_confusion_matrix.png → mlartifacts/.../figures/
15:28:30 | INFO | Artefato logado: dummy_roc_curve.png → mlartifacts/.../figures/
2026/04/21 15:28:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 15:28:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_per

### 4.7 Sumário — DummyClassifier

Tabela de referência para comparação com os modelos subsequentes.


In [138]:
"""Tabela de resultados do DummyClassifier — referência para comparação futura."""
dummy_summary_df = (
    pd.DataFrame(
        [{
            "Modelo"    : "DummyClassifier (most_frequent)",
            "AUC-ROC"   : dummy_hold_out["test_roc_auc"],
            "PR-AUC"    : dummy_hold_out["test_average_precision"],
            "F1"        : dummy_hold_out["test_f1"],
            "Recall"    : dummy_hold_out["test_recall"],
            "Precision" : dummy_hold_out["test_precision"],
            "Accuracy"  : dummy_hold_out["test_accuracy"],
            "Valor Líq. (R$)" : dummy_biz["biz_valor_liquido"],
        }]
    )
    .set_index("Modelo")
    .round(4)
)

logger.info("\n%s", dummy_summary_df.T.to_string())
dummy_summary_df.T

15:28:36 | INFO | 
Modelo           DummyClassifier (most_frequent)
AUC-ROC                                     0.50
PR-AUC                                      0.26
F1                                          0.00
Recall                                      0.00
Precision                                   0.00
Accuracy                                    0.74
Valor Líq. (R$)                      -1058340.00


Modelo,DummyClassifier (most_frequent)
AUC-ROC,0.50
PR-AUC,0.26
F1,0.00
Recall,0.00
Precision,0.00
Accuracy,0.74
Valor Líq. (R$),-1058340.00


#### Interpretação do DummyClassifier

| Chave MLflow | Valor esperado | Significado |
|---|---|---|
| `test_roc_auc` | 0.500 | Equivalente ao acaso — piso absoluto |
| `test_average_precision` | ≈ 0.265 | Igual à taxa de churn — piso da PR-AUC |
| `test_f1` | 0.000 | Nenhum churner detectado |
| `test_recall` | 0.000 | 100% dos FN — perda máxima de negócio |
| `test_accuracy` | ≈ 0.735 | ⚠️ Enganosa — apenas reflete a classe dominante |
| `biz_valor_liquido_r$` | negativo máximo | Pior cenário: todos os churners perdidos |


---
## 5. Modelo B — Regressão Logística

A `LogisticRegression` serve como **teto linear** — o melhor resultado alcançável
com um modelo linear sem otimização adicional.

**Decisões de design:**
- `class_weight="balanced"` — compensa o desbalanceamento residual (~26% churn no test)
- `solver="lbfgs"` — eficiente para datasets de tamanho médio (~5-6k amostras), suporta `class_weight`
- `max_iter=1000` — garante convergência sem `ConvergenceWarning`
- `C=1.0` — regularização padrão (sem tuning — este é um baseline, não o modelo final)

Além das métricas técnicas, realizamos:
1. **Análise de coeficientes** — identifica as features mais relevantes para o churn
2. **Otimização de threshold** — busca o ponto de corte que maximiza o valor de negócio
3. **Curva PR-AUC** — mais informativa que ROC em datasets desbalanceados


### 5.1 Definição do Pipeline

Parâmetros declarados explicitamente em `LOGISTIC_PARAMS` para rastreabilidade total no MLflow.
O pipeline encapsula o modelo para compatibilidade com futuras etapas de pré-processamento.


In [139]:
LOGISTIC_PARAMS: dict = {
    "class_weight" : "balanced",
    "max_iter"     : 1000,
    "random_state" : RANDOM_STATE,
    "solver"       : "lbfgs",
    "C"            : 1.0,
}

logger.info("LOGISTIC_PARAMS: %s", LOGISTIC_PARAMS)

logistic_model = LogisticRegression(**LOGISTIC_PARAMS)

logistic_pipeline = Pipeline(
    [("classifier", logistic_model)]
)

15:28:36 | INFO | LOGISTIC_PARAMS: {'class_weight': 'balanced', 'max_iter': 1000, 'random_state': 42, 'solver': 'lbfgs', 'C': 1.0}


### 5.2 Validação Cruzada Estratificada (5-fold)

Usa o mesmo `CV` e `SCORING` da seção anterior — garantindo comparabilidade direta
entre Dummy e Logística no MLflow.

Os resultados de CV medem a **estabilidade** do modelo no train set (pós-SMOTE+ENN).
Os desvios-padrão indicam variância entre folds — valores altos sugerem overfitting ou
instabilidade de treinamento.


In [140]:
logger.info(
    "cross_validate [Logística] | solver=%s | C=%.2f | folds=%d | stratified=True",
    LOGISTIC_PARAMS["solver"],
    LOGISTIC_PARAMS["C"],
    CV.n_splits,
)

logistic_cv_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=CV,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)

logistic_cv_summary: dict[str, dict[str, float]] = {}
for key, values in logistic_cv_results.items():
    if key.startswith("test_"):
        name = key.removeprefix("test_")
        logistic_cv_summary[name] = {
            "mean" : float(values.mean()),
            "std"  : float(values.std()),
        }
        logger.info(
            "  cv [Logística] %-26s  mean=%+.4f  std=%.4f",
            name, values.mean(), values.std(),
        )

15:28:36 | INFO | cross_validate [Logística] | solver=lbfgs | C=1.00 | folds=5 | stratified=True
15:28:39 | INFO |   cv [Logística] roc_auc                     mean=+0.8639  std=0.0172
15:28:39 | INFO |   cv [Logística] average_precision           mean=+0.6914  std=0.0312
15:28:39 | INFO |   cv [Logística] f1                          mean=+0.6523  std=0.0169
15:28:39 | INFO |   cv [Logística] recall                      mean=+0.8195  std=0.0262
15:28:39 | INFO |   cv [Logística] precision                   mean=+0.5418  std=0.0122


### 5.3 Treino Final e Predições no Hold-out

Treina no `X_train` completo (todos os 5 folds) e gera predições no `X_test`.

> ⚠️ Esta é a **única avaliação** no test set — resultados não podem ser usados
> para tuning de hiperparâmetros (isso seria data leakage).


In [141]:
# Treino final no X_train completo
logistic_pipeline.fit(X_train, y_train)

logistic_y_pred  = logistic_pipeline.predict(X_test)
logistic_y_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

# Extrai o modelo do pipeline para acesso aos coeficientes
fitted_logistic: LogisticRegression = logistic_pipeline.named_steps["classifier"]

logger.info(
    "Treino [Logística] finalizado | n_iter_=%d | convergiu=%s",
    fitted_logistic.n_iter_[0],
    fitted_logistic.n_iter_[0] < LOGISTIC_PARAMS["max_iter"],
)

15:28:40 | INFO | Treino [Logística] finalizado | n_iter_=66 | convergiu=True


### 5.4 Avaliação no Hold-out — Métricas Técnicas

Calcula o conjunto completo de métricas usando a função `compute_holdout_metrics`.
A avaliação usa threshold padrão de 0.5 — o threshold ótimo será calculado na seção 5.6.


In [142]:
logistic_hold_out = compute_holdout_metrics(
    y_test.values, logistic_y_pred, logistic_y_proba
)

for k, v in logistic_hold_out.items():
    logger.info("  hold-out [Logística] %-30s  %.4f", k, v)

15:28:40 | INFO |   hold-out [Logística] test_roc_auc                    0.8500
15:28:40 | INFO |   hold-out [Logística] test_average_precision          0.6700
15:28:40 | INFO |   hold-out [Logística] test_f1                         0.6200
15:28:40 | INFO |   hold-out [Logística] test_recall                     0.7800
15:28:40 | INFO |   hold-out [Logística] test_precision                  0.5100
15:28:40 | INFO |   hold-out [Logística] test_accuracy                   0.7500
15:28:40 | INFO |   hold-out [Logística] test_tp                         289.0000
15:28:40 | INFO |   hold-out [Logística] test_tn                         758.0000
15:28:40 | INFO |   hold-out [Logística] test_fp                         275.0000
15:28:40 | INFO |   hold-out [Logística] test_fn                         83.0000


### 5.5 Figuras — Logística

Gera 4 figuras para a Regressão Logística:
1. **Confusion Matrix** — distribuição de TP/TN/FP/FN com threshold=0.5
2. **ROC Curve** — AUC-ROC com diagonal de referência (acaso)
3. **PR-AUC Curve** — Precision-Recall, mais informativa em dados desbalanceados
4. **Feature Importance** — magnitude dos coeficientes (|coef|), top 20 features

Todas salvas em `reports/figures/baselines/` (commitadas no git) antes do registro no MLflow.


In [143]:
# ── 5.5.1 Confusion Matrix ────────────────────────────────────────────────────
logistic_cm_path = save_confusion_matrix(
    y_true=y_test,
    y_pred=logistic_y_pred,
    path=FIGURES_DIR / "logistic_confusion_matrix.png",
    title="Confusion Matrix — Logistic Regression (thr=0.5)",
)

# ── 5.5.2 ROC Curve ───────────────────────────────────────────────────────────
logistic_roc_path = save_roc_curve(
    y_true=y_test,
    y_proba=logistic_y_proba,
    path=FIGURES_DIR / "logistic_roc_curve.png",
    model_name="Logistic Regression",
    title=f"ROC Curve — Logistic Regression  (AUC = {logistic_hold_out['test_roc_auc']:.3f})",
)

# ── 5.5.3 PR-AUC Curve ────────────────────────────────────────────────────────
pr_precisions, pr_recalls, pr_thresholds = precision_recall_curve(y_test, logistic_y_proba)

# Threshold ótimo por F1 (para marcar na curva — threshold de negócio calculado na seção 5.6)
_f1_arr = (
    2 * pr_precisions[:-1] * pr_recalls[:-1]
    / (pr_precisions[:-1] + pr_recalls[:-1] + 1e-9)
)
_best_f1_thr = float(pr_thresholds[np.argmax(_f1_arr)])

logistic_pr_path = save_precision_recall_curve(
    precision_arr=pr_precisions,
    recall_arr=pr_recalls,
    thresholds=pr_thresholds,
    best_threshold=_best_f1_thr,
    path=FIGURES_DIR / "logistic_pr_curve.png",
    model_name="Logistic Regression",
)

# ── 5.5.4 Feature Importance — |coeficientes| ─────────────────────────────────
coef_abs = np.abs(fitted_logistic.coef_.ravel())
logistic_fi_path = save_feature_importance(
    importances=coef_abs,
    feature_names=feature_names,
    path=FIGURES_DIR / "logistic_feature_importance.png",
    title="Feature Importance — |Coeficientes| Logistic Regression",
    color="steelblue",
    top_n=20,
)

# ── 5.5.5 Threshold × F1/Recall ───────────────────────────────────────────────
logistic_thr_path = save_threshold_f1_recall(
    thresholds=pr_thresholds,
    f1_arr=_f1_arr,
    recall_arr=pr_recalls,
    best_threshold=_best_f1_thr,
    path=FIGURES_DIR / "logistic_threshold_f1_recall.png",
)

logger.info("Figuras [Logística] salvas em: %s", FIGURES_DIR.relative_to(PROJECT_ROOT))

15:28:40 | INFO | Figuras [Logística] salvas em: reports\figures\baselines


### 5.6 Análise de Coeficientes (Feature Importance)

Os coeficientes da Regressão Logística têm interpretação direta:
- **Coeficiente positivo** → feature aumenta a probabilidade de churn
- **Coeficiente negativo** → feature reduz a probabilidade de churn
- **|coeficiente| maior** → maior impacto na predição (features mais relevantes)

Esses coeficientes serão registrados como JSON no MLflow para análise comparativa com a MLP.


In [144]:
# Dicionário de coeficientes para log no MLflow
coef_dict: dict[str, float] = {
    name: float(round(coef, 6))
    for name, coef in zip(feature_names, fitted_logistic.coef_.ravel())
}

# Top 10 por magnitude
top10 = sorted(coef_dict.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
logger.info("Top 10 features por |coeficiente|:")
for feat, val in top10:
    logger.info("  %-35s : %+.4f", feat, val)

# Série ordenada para exibição
coef_series = (
    pd.Series(coef_dict)
    .reindex(pd.Series(coef_dict).abs().sort_values(ascending=False).index)
)

logger.info("\nTop 10 coeficientes (magnitude):\n%s", coef_series.head(10).to_string())

15:28:40 | INFO | Top 10 features por |coeficiente|:
15:28:40 | INFO |   internet_service_No                 : -1.6147
15:28:40 | INFO |   dependents                          : -1.5814
15:28:40 | INFO |   total_charges                       : -1.1850
15:28:40 | INFO |   contract_Two year                   : -0.9854
15:28:40 | INFO |   phone_service                       : -0.7510
15:28:40 | INFO |   is_month_to_month                   : +0.7010
15:28:40 | INFO |   internet_service_Fiber optic        : +0.6496
15:28:40 | INFO |   is_fiber_optic                      : +0.6496
15:28:40 | INFO |   has_security_support                : -0.4829
15:28:40 | INFO |   multiple_lines                      : +0.4459
15:28:40 | INFO | 
Top 10 coeficientes (magnitude):
internet_service_No            -1.614717
dependents                     -1.581395
total_charges                  -1.185036
contract_Two year              -0.985373
phone_service                  -0.750979
is_month_to_month             

### 5.7 Threshold Ótimo por Valor de Negócio

Busca o threshold que **maximiza o valor líquido de negócio** (R$) no hold-out.

**Por que isso importa?**  
O threshold padrão de 0.5 é arbitrário. Em problemas de churn:
- Um FN (cliente perdido) custa aproximadamente $ 2.845,00
- Um FP (campanha desnecessária) custa $ 73,52
- Portanto, vale a pena reduzir o threshold para capturar mais churners (aumentar Recall),
  mesmo que isso aumente os FPs — desde que o trade-off seja favorável.


In [145]:
# Busca exaustiva: threshold ótimo que maximiza valor líquido de negócio
logistic_best_thr, logistic_best_value, logistic_thr_results = find_best_threshold_business(
    y_true=y_test.values,
    y_proba=logistic_y_proba,
)

# Métricas com threshold ótimo
logistic_y_pred_opt = (logistic_y_proba >= logistic_best_thr).astype(int)
logistic_hold_out_opt = compute_holdout_metrics(
    y_test.values, logistic_y_pred_opt, logistic_y_proba
)
logistic_biz_opt = compute_business_metrics(
    tp=logistic_hold_out_opt["test_tp"],
    fp=logistic_hold_out_opt["test_fp"],
    fn=logistic_hold_out_opt["test_fn"],
)

logger.info(
    "Threshold ótimo [Logística] | thr=%.4f | F1=%.4f | Recall=%.4f | Valor_Líq=R$%.2f",
    logistic_best_thr,
    logistic_hold_out_opt["test_f1"],
    logistic_hold_out_opt["test_recall"],
    logistic_biz_opt["biz_valor_liquido"],
)

# Comparação: thr=0.5 vs thr ótimo
logistic_biz_default = compute_business_metrics(
    tp=logistic_hold_out["test_tp"],
    fp=logistic_hold_out["test_fp"],
    fn=logistic_hold_out["test_fn"],
)

ganho_threshold = logistic_biz_opt["biz_valor_liquido"] - logistic_biz_default["biz_valor_liquido"]
logger.info("Ganho pela otimização de threshold: R$ %.2f", ganho_threshold)

15:28:40 | INFO | find_best_threshold_business | best_thr=0.0937 | best_value=R$968734.72
15:28:40 | INFO | Threshold ótimo [Logística] | thr=0.0937 | F1=0.5200 | Recall=0.9900 | Valor_Líq=R$968734.72
15:28:40 | INFO | Ganho pela otimização de threshold: R$ 424130.00


### 5.8 Registro no MLflow — Logística

Run separado do Dummy — permite comparação lado a lado na UI do MLflow.

Além do que é registrado no run do Dummy, este run inclui:
- **Coeficientes JSON** — `baselines/logistic/coefficients.json`
- **Curva PR-AUC** — `figures/logistic_pr_curve.png`
- **Feature importance** — `figures/logistic_feature_importance.png`
- **Threshold ótimo** — `logistic_best_threshold` (parâmetro) + métricas com sufixo `_opt`
- **Modelo registrado** no Model Registry como `"baseline-logistic"`


In [146]:
with mlflow_run(run_name="baseline_logistic_regression") as run_logistic:
    LOGISTIC_RUN_ID = run_logistic.info.run_id
    LOGISTIC_EXP_ID = run_logistic.info.experiment_id
    logger.info("Run [Logística] iniciado | run_id=%s | exp_id=%s", LOGISTIC_RUN_ID, LOGISTIC_EXP_ID)

    # ── 1. Tags ───────────────────────────────────────────────────────────────
    mlflow.set_tags({
        "notebook" : "3.00",
        "fase"     : "baseline",
        "modelo"   : "logistic",
        "task"     : "classification",
        "framework": "sklearn",
    })

    # ── 2. Dataset versionado (URI + MD5) ─────────────────────────────────────
    log_dataset_to_mlflow(X_train, y_train, split="train", source_path=TRAIN_PATH)
    log_dataset_to_mlflow(X_test,  y_test,  split="test",  source_path=TEST_PATH)
    logger.info("Datasets logados com hash MD5")

    # ── 3. Parâmetros ─────────────────────────────────────────────────────────
    mlflow.log_params({
        # Modelo
        "model_type"        : "LogisticRegression",
        **LOGISTIC_PARAMS,
        "feature_selection" : "none",
        # Threshold
        "default_threshold" : 0.5,
        "best_threshold_biz": round(logistic_best_thr, 4),
        # Validação
        "cv_folds"          : CV.n_splits,
        "cv_stratified"     : True,
        "cv_scoring"        : str(SCORING),
        # Dados
        "dataset_train"     : str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
        "dataset_test"      : str(TEST_PATH.relative_to(PROJECT_ROOT)),
        "n_train"           : int(len(X_train)),
        "n_test"            : int(len(X_test)),
        "n_features"        : int(X_train.shape[1]),
        "churn_rate_train"  : round(float(y_train.mean()), 4),
        "churn_rate_test"   : round(float(y_test.mean()),  4),
        "target_col"        : TARGET,
        # Custos de negócio
        "cost_fn"        : COST_FN,
        "cost_fp"        : COST_FP,
        "cost_clv"       : COST_CLV,
        "cost_retention" : COST_RETENTION,
    })
    logger.info("Parâmetros logados")

    # ── 4. Métricas CV ────────────────────────────────────────────────────────
    for scorer_name, stats in logistic_cv_summary.items():
        mlflow.log_metric(f"cv_{scorer_name}_mean", stats["mean"])
        mlflow.log_metric(f"cv_{scorer_name}_std",  stats["std"])
    logger.info("Métricas CV logadas (%d scorers × 2)", len(logistic_cv_summary))

    # ── 5. Métricas hold-out técnicas (thr=0.5) ───────────────────────────────
    for metric_name, metric_value in logistic_hold_out.items():
        mlflow.log_metric(metric_name, metric_value)
    logger.info("Métricas hold-out (thr=0.5) logadas (%d)", len(logistic_hold_out))

    # ── 6. Métricas hold-out com threshold ótimo (sufixo _opt) ────────────────
    for metric_name, metric_value in logistic_hold_out_opt.items():
        mlflow.log_metric(f"{metric_name}_opt", metric_value)
    logger.info("Métricas hold-out (thr ótimo) logadas (%d)", len(logistic_hold_out_opt))

    # ── 7. Métricas de negócio (thr=0.5 e thr ótimo) ─────────────────────────
    for metric_name, metric_value in logistic_biz_default.items():
        mlflow.log_metric(metric_name, metric_value)
    for metric_name, metric_value in logistic_biz_opt.items():
        mlflow.log_metric(f"{metric_name}_opt", metric_value)
    mlflow.log_metric("biz_ganho_threshold", ganho_threshold)
    logger.info("Métricas de negócio logadas")

    # ── 8. Artefatos visuais ──────────────────────────────────────────────────
    for fig_path in [
        logistic_cm_path,
        logistic_roc_path,
        logistic_pr_path,
        logistic_fi_path,
        logistic_thr_path,
    ]:
        mlflow.log_artifact(str(fig_path), artifact_path="figures")
        logger.info("Artefato logado: %s → mlartifacts/.../figures/", fig_path.name)

    # ── 9. Coeficientes como JSON ─────────────────────────────────────────────
    mlflow.log_dict(coef_dict, "baselines/logistic/coefficients.json")
    logger.info("Coeficientes logados como JSON (%d features)", len(coef_dict))

    # ── 10. Registro no Model Registry ───────────────────────────────────────
    model_info = mlflow.sklearn.log_model(
        sk_model=logistic_pipeline,
        artifact_path="model",
        registered_model_name="baseline-logistic",
        input_example=X_test.iloc[:5],
    )

    logger.info("Model URI : %s", model_info.model_uri)
    logger.info(
        "Run [Logística] finalizado | experiment=%s | run_id=%s",
        MLFLOW_EXPERIMENT, LOGISTIC_RUN_ID,
    )
    logger.info(
        "Resumo [Logística] | roc_auc=%.4f | pr_auc=%.4f | f1=%.4f | recall=%.4f | valor_liquido_opt=R$%.2f",
        logistic_hold_out["test_roc_auc"],
        logistic_hold_out["test_average_precision"],
        logistic_hold_out["test_f1"],
        logistic_hold_out["test_recall"],
        logistic_biz_opt["biz_valor_liquido"],
    )

15:28:40 | INFO | Run [Logística] iniciado | run_id=30fbe46bff884804877fe89cd44e07d2 | exp_id=1
15:28:40 | INFO | Datasets logados com hash MD5
15:28:40 | INFO | Parâmetros logados
15:28:40 | INFO | Métricas CV logadas (5 scorers × 2)
15:28:41 | INFO | Métricas hold-out (thr=0.5) logadas (10)
15:28:41 | INFO | Métricas hold-out (thr ótimo) logadas (10)
15:28:41 | INFO | Métricas de negócio logadas
15:28:41 | INFO | Artefato logado: logistic_confusion_matrix.png → mlartifacts/.../figures/
15:28:41 | INFO | Artefato logado: logistic_roc_curve.png → mlartifacts/.../figures/
15:28:41 | INFO | Artefato logado: logistic_pr_curve.png → mlartifacts/.../figures/
15:28:41 | INFO | Artefato logado: logistic_feature_importance.png → mlartifacts/.../figures/
15:28:41 | INFO | Artefato logado: logistic_threshold_f1_recall.png → mlartifacts/.../figures/
15:28:41 | INFO | Coeficientes logados como JSON (30 features)
2026/04/21 15:28:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please

Registered model 'baseline-logistic' already exists. Creating a new version of this model...
Created version '4' of model 'baseline-logistic'.
15:28:45 | INFO | Model URI : models:/m-31b5437a408d4b32b08977d8e3243130
15:28:45 | INFO | Run [Logística] finalizado | experiment=churn-telecom | run_id=30fbe46bff884804877fe89cd44e07d2
15:28:45 | INFO | Resumo [Logística] | roc_auc=0.8500 | pr_auc=0.6700 | f1=0.6200 | recall=0.7800 | valor_liquido_opt=R$968734.72


### 5.9 Sumário — Regressão Logística


In [147]:
"""Tabela de resultados da Regressão Logística (thr=0.5 e thr ótimo)."""
logistic_summary_df = (
    pd.DataFrame(
        [
            {
                "Modelo"         : "Logistic Regression (thr=0.5)",
                "AUC-ROC"        : logistic_hold_out["test_roc_auc"],
                "PR-AUC"         : logistic_hold_out["test_average_precision"],
                "F1"             : logistic_hold_out["test_f1"],
                "Recall"         : logistic_hold_out["test_recall"],
                "Precision"      : logistic_hold_out["test_precision"],
                "Accuracy"       : logistic_hold_out["test_accuracy"],
                "Valor Líq. (R$)": logistic_biz_default["biz_valor_liquido"],
            },
            {
                "Modelo"         : f"Logistic Regression (thr={logistic_best_thr:.3f}, ótimo biz)",
                "AUC-ROC"        : logistic_hold_out_opt["test_roc_auc"],
                "PR-AUC"         : logistic_hold_out_opt["test_average_precision"],
                "F1"             : logistic_hold_out_opt["test_f1"],
                "Recall"         : logistic_hold_out_opt["test_recall"],
                "Precision"      : logistic_hold_out_opt["test_precision"],
                "Accuracy"       : logistic_hold_out_opt["test_accuracy"],
                "Valor Líq. (R$)": logistic_biz_opt["biz_valor_liquido"],
            },
        ]
    )
    .set_index("Modelo")
    .round(4)
)

logger.info("\n%s", logistic_summary_df.T.to_string())
logistic_summary_df.T

15:28:45 | INFO | 
Modelo           Logistic Regression (thr=0.5)  Logistic Regression (thr=0.094, ótimo biz)
AUC-ROC                                   0.85                                        0.85
PR-AUC                                    0.67                                        0.67
F1                                        0.62                                        0.52
Recall                                    0.78                                        0.99
Precision                                 0.51                                        0.35
Accuracy                                  0.75                                        0.50
Valor Líq. (R$)                      544604.72                                   968734.72


Modelo,Logistic Regression (thr=0.5),"Logistic Regression (thr=0.094, ótimo biz)"
AUC-ROC,0.85,0.85
PR-AUC,0.67,0.67
F1,0.62,0.52
Recall,0.78,0.99
Precision,0.51,0.35
Accuracy,0.75,0.50
Valor Líq. (R$),544604.72,968734.72


---
## 6. Comparativo Final: Dummy vs Logística

Tabela consolidada de todos os baselines para referência nas etapas 2 e 3 do Tech Challenge.

> **Critério de sucesso**: qualquer modelo subsequente (MLP, ensembles) deve superar a
> Regressão Logística com threshold ótimo em **pelo menos** AUC-ROC, PR-AUC e Valor Líquido de Negócio.


In [148]:
"""Tabela comparativa consolidada — Dummy × Logística (thr=0.5) × Logística (thr ótimo)."""
final_comparison = pd.concat(
    [dummy_summary_df, logistic_summary_df],
    axis=0,
)

logger.info("\n=== COMPARATIVO FINAL DOS BASELINES ===\n%s", final_comparison.T.to_string())

# ── Destaque: ganho da Logística sobre o Dummy ─────────────────────────────────
delta_auc = (
    logistic_hold_out["test_roc_auc"] - dummy_hold_out["test_roc_auc"]
)
delta_prauc = (
    logistic_hold_out["test_average_precision"] - dummy_hold_out["test_average_precision"]
)
delta_biz = (
    logistic_biz_opt["biz_valor_liquido"] - dummy_biz["biz_valor_liquido"]
)

logger.info("\n--- Ganho Logística (thr ótimo) vs Dummy ---")
logger.info("  ΔAUC-ROC          : %+.4f", delta_auc)
logger.info("  ΔPR-AUC           : %+.4f", delta_prauc)
logger.info("  ΔValor Líq.  (R$) : %+.2f", delta_biz)

final_comparison.T

15:28:45 | INFO | 
=== COMPARATIVO FINAL DOS BASELINES ===
Modelo           DummyClassifier (most_frequent)  Logistic Regression (thr=0.5)  Logistic Regression (thr=0.094, ótimo biz)
AUC-ROC                                     0.50                           0.85                                        0.85
PR-AUC                                      0.26                           0.67                                        0.67
F1                                          0.00                           0.62                                        0.52
Recall                                      0.00                           0.78                                        0.99
Precision                                   0.00                           0.51                                        0.35
Accuracy                                    0.74                           0.75                                        0.50
Valor Líq. (R$)                      -1058340.00                      544

Modelo,DummyClassifier (most_frequent),Logistic Regression (thr=0.5),"Logistic Regression (thr=0.094, ótimo biz)"
AUC-ROC,0.50,0.85,0.85
PR-AUC,0.26,0.67,0.67
F1,0.00,0.62,0.52
Recall,0.00,0.78,0.99
Precision,0.00,0.51,0.35
Accuracy,0.74,0.75,0.50
Valor Líq. (R$),-1058340.00,544604.72,968734.72


# Interpretação Final

| Métrica            | Dummy | Logística (thr=0.5) | Logística (thr ótimo) | Referência MLP |
|-------------------|-------|---------------------|------------------------|----------------|
| AUC-ROC           | 0.50  | 0.85                | 0.85                   | deve superar   |
| PR-AUC            | 0.26  | 0.67                | 0.67                   | deve superar   |
| Recall            | 0.00  | 0.78                | 0.99                   | deve superar   |
| Valor Líq. (R$)   | -1.058.340,00 | 544.604,72 | 968.734,72            | deve superar   |

Os valores da Logística foram consolidados a partir da execução dos baselines, destacando o impacto financeiro da otimização do threshold.

---

# Referência de Artefatos MLflow

| Run Name                     | Métricas principais |
|-----------------------------|----------------------|
| dummy_most_frequent         | AUC-ROC=0.50, Recall=0.00, Valor Líq.=-1.05M |
| baseline_logistic_regression| AUC-ROC=0.85, PR-AUC=0.67, thr_ótimo=0.094, Valor Líq.=+0,97M |

---

Para visualizar no painel:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
Ou então pelo python:

```python
python -m mlflow ui
```